# PertCF — South German Credit Dataset
## Reproducing Table 2 from Bayrak & Bach (SGAI 2023)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/b-bayrak/PertCF-Explainer/blob/main/examples/quickstart_german_credit.ipynb)
[![PyPI](https://img.shields.io/pypi/v/pertcf.svg)](https://pypi.org/project/pertcf/)

This notebook reproduces the benchmark results from **Table 2** of:

> Bayrak, B. & Bach, K. (2023). *PertCF: A Perturbation-Based Counterfactual Generation Approach.*  
> SGAI AI-2023, LNAI 14381, pp. 174–187. https://doi.org/10.1007/978-3-031-47994-6_13

**Dataset:** South German Credit, 1,000 instances, 21 features (3 numeric, 18 categorical), binary classification.
**Model:** Gradient Boosting Classifier (~0.81 accuracy, matching the paper).


In [ ]:
# Install pertcf (no Java, no REST server required)
%pip install -q pertcf

## 1. Load the South German Credit Dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from pertcf import PertCFExplainer
from pertcf import metrics

# Public dataset from UCI
url = ("https://raw.githubusercontent.com/uci-ml-repo/ucimlrepo"
       "/main/datasets/south_german_credit/SouthGermanCredit.asc")
df = pd.read_csv(url, sep=" ")
print(f"Dataset: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(3)

## 2. Train a Gradient Boosting Classifier

In [ ]:
LABEL = "kredit"
CATEGORICAL_FEATURES = [
    "laufkont", "moral", "verw", "sparkont", "beszeit", "rate",
    "famges", "buerge", "wohnzeit", "verm", "weitkred", "wohn",
    "bishkred", "beruf", "pers", "telef", "gastarb",
]

X = df.drop(columns=[LABEL])
y = df[LABEL].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
clf = GradientBoostingClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
acc = clf.score(X_test, y_test)
print(f"Accuracy: {acc:.3f}  (paper reports ~0.81)")

## 3. Fit the PertCF Explainer

In [ ]:
explainer = PertCFExplainer(
    model=clf,
    X_train=X_train,
    y_train=y_train,
    categorical_features=CATEGORICAL_FEATURES,
    label=LABEL,
    num_iter=5,   # paper setting
    coef=5.0,     # paper setting
)
explainer.fit(verbose=True)

## 4. Explain a Single "Bad Credit" Prediction

We pick a test instance predicted as *bad credit* (class 0) and ask:  
**"What would need to change for this person to be approved?"**


In [ ]:
# Find a "bad credit" instance
bad_mask = y_test == "0"
instance = X_test[bad_mask].iloc[0].copy()
instance[LABEL] = "0"

cf = explainer.explain(instance)

print("=== Original instance ===")
print(instance[["laufkont", "laufzeit", "moral", "betrag", "verw"]].to_string())
print()
print("=== Counterfactual ===")
print(cf[["laufkont", "laufzeit", "moral", "betrag", "verw"]].to_string())
print(f"\nClass: {instance[LABEL]} → {cf[LABEL]}")

### What changed?

In [ ]:
changes = {
    feat: {"original": instance[feat], "counterfactual": cf[feat]}
    for feat in explainer.feature_names
    if str(instance[feat]) != str(cf[feat])
}
pd.DataFrame(changes).T.rename_axis("feature")

### Visualise numeric feature changes

In [ ]:
import matplotlib.pyplot as plt

numeric_feats = [f for f in explainer.feature_names
                 if f not in explainer.categorical_features]

orig_vals = [float(instance[f]) for f in numeric_feats]
cf_vals   = [float(cf[f])       for f in numeric_feats]

x = np.arange(len(numeric_feats))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 3))
bars1 = ax.bar(x - width/2, orig_vals, width, label="Original",        color="#e05252")
bars2 = ax.bar(x + width/2, cf_vals,   width, label="Counterfactual",  color="#4caf50")
ax.set_xticks(x)
ax.set_xticklabels(numeric_feats, rotation=20, ha="right")
ax.set_ylabel("Feature value")
ax.set_title("Original vs Counterfactual — numeric features")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Benchmark — Reproducing Table 2

Paper results (South German Credit, `num_iter=5`, `coef=5`):

| Metric | PertCF | DiCE | CF-SHAP |
|--------|--------|------|---------|
| Dissimilarity ↓ | **0.0517** | 0.0557 | 0.2555 |
| Sparsity | 0.7983 | 0.9111 | 0.5842 |
| Instability ↓ | **0.0518** | 0.0560 | 0.2555 |


In [ ]:
X_test_labeled = X_test.assign(**{LABEL: y_test})
results = explainer.benchmark(X_test_labeled, n=50, coef=5.0, verbose=True)

print("\n--- Comparison ---")
comparison = pd.DataFrame({
    "Paper (PertCF)": {"dissimilarity": 0.0517, "sparsity": 0.7983},
    "Reproduced":     {"dissimilarity": results["dissimilarity"],
                       "sparsity":     results["sparsity"]},
})
comparison.round(4)

## 6. Inspect SHAP Feature Weights

In [ ]:
import matplotlib.pyplot as plt

shap_df = explainer._shap_weights.drop(columns=[], errors="ignore")

fig, axes = plt.subplots(1, len(shap_df), figsize=(14, 4), sharey=True)
if len(shap_df) == 1:
    axes = [axes]

for ax, (cls, row) in zip(axes, shap_df.iterrows()):
    top = row.nlargest(8)
    ax.barh(top.index[::-1], top.values[::-1], color="#5c85d6")
    ax.set_title(f"Class {cls}")
    ax.set_xlabel("Normalised SHAP weight")

fig.suptitle("SHAP Feature Weights per Class", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("These weights drive PertCF's similarity function.")
print("Features with higher weight have more influence on the CF direction.")